# SOLiD-PINN v0.9.0d — Adaptive coupled closure
CPU · Run all · No Drive · No token. Up to 306 fits. Synthetic reduced coupled x-r model only. The 10% marginal false activation target assumes exchangeability under the specified null simulator; it is not an empirical guarantee on real cells.
## 1. Pinned sources and eight regression tests


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess,sys,os,tempfile,json,hashlib,math
REV="f9aded427d46bfbd9b0ea829d3e909613aaa9d33"
repo=Path(tempfile.mkdtemp(prefix="solid-v090d-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
hashes={"adaptive_coupled.py":"7f46443b8f09199b9c96f12197555408e81783c8f5733cd19fcec4b3395c7256","test_adaptive_coupled.py":"a7a01301e63ff54129aa2090ab83471a6de536a0145b86ace4b6605bdaa6bb2d","coupled_bridge.py":"87f474918e731710193a4b01ee2075d5cc6a157e12fc1c9e1fdcf3779b37d3e3","test_coupled_bridge.py":"8d1e7a81d5cb10c77338e4fab2543da146fc103e8a60dfe463df5ed35424d2de","hybrid_p2d.py":"20aaeede052a29a2b65b1cd322b9fade3b4094fd4e1564d15787d7e240c51f18"}
for name,digest in hashes.items(): assert hashlib.sha256((repo/"scripts"/name).read_bytes()).hexdigest()==digest
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_coupled_bridge.py","scripts/test_adaptive_coupled.py"],cwd=repo,env=env,check=True)


## 2. Prospective validation and independent test
38 calibration cases and 64 held-out cases; three candidate models per case. Independent train/validation noise; separate voltage validation profiles. Gate uses normalized validation MSE improvement; no held-out test metric enters the decision. Report reaction current density error against aggregated fine reference. Preserve failed gates.


In [ ]:
out=repo/"results.json"
subprocess.run([sys.executable,"-u","scripts/adaptive_coupled.py","--out",str(out)],cwd=repo,env=env,check=True)
d=json.loads(out.read_text())
assert len(d["numerical_checks"])==9
assert d["numerical_gate_passed"]==all(x["passed"] for x in d["numerical_checks"])
methods=("calibrated","linear","rbf");truths=("null","rest","peak","weak_rest")
if d["complete"]:
    assert d["numerical_gate_passed"]
    assert len(d["calibration"])==38 and len(d["test"])==64
    assert {(x["noise"],x["truth"],x["seed"]) for x in d["calibration"]}=={(n,"null",s) for n in (1.,5.) for s in range(1000,1019)}
    assert {(x["noise"],x["truth"],x["seed"]) for x in d["test"]}=={(n,t,s) for n in (1.,5.) for t in truths for s in range(2000,2008)}
for x in d["calibration"]+d["test"]:
    if x.get("fit_gate_passed"):
        assert x["truth_gate_passed"] and max(x["mesh_delta_mv"].values())<=1
        assert set(x["models"])==set(methods)
        assert all(m["success"] and m["physical_checks_passed"] for m in x["models"].values())
        s=x["validation_scores"];best=min(methods[1:],key=s.get)
        assert math.isclose(x["gain"],s["calibrated"]-s[best],abs_tol=1e-12)
        assert x["raw"]==(best if x["gain"]>0 else "calibrated")
        if "gated" in x: assert x["gated"]==(best if x["gain"]>d["thresholds"][str(x["noise"])] else "calibrated")
for n,q in d["thresholds"].items():
    s=sorted(x["gain"] for x in d["calibration"] if x["noise"]==float(n))
    assert len(s)==19 and q==max(0.,s[17])
d.update(source_revision=REV,source_hashes=hashes,targeted_tests_passed=8)
print("Complete:",d["complete"],"Calibration:",len(d["calibration"]),"Test:",len(d["test"]))


## 3. Export evidence
Download the hash-addressed JSON before ending the runtime. No persistent Drive storage.


In [ ]:
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/SOLiD-PINN/v090d/handoff");root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v090d_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    tmp=dest.with_suffix(".part");tmp.write_bytes(raw);tmp.replace(dest)
assert dest.read_bytes()==raw
print("LOCAL FILE VERIFIED",dest.name,sha)
try: files.download(str(dest))
except Exception as e: print("Download manually from Files panel:",str(dest),str(e))
